# T-Series Forecasting — Bronze Layer
## Databricks Medallion Architecture + Delta MERGE

**Catalog:** `workspace`  
**Bronze schema:** `workspace.bronze`  
**Volume:** `workspace.bronze.tseries`

This notebook:
- Reads the five supplied CSV files from the Databricks Volume.
- Creates Bronze Delta tables.
- Uses **Delta `MERGE INTO`** for idempotent/upsert ingestion.
- Preserves the raw business columns while adding ingestion metadata.

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

CATALOG = "workspace"
BRONZE = f"{CATALOG}.bronze"
VOLUME = f"/Volumes/{CATALOG}/bronze/tseries"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {BRONZE}")
display(dbutils.fs.ls(VOLUME))

### Expected files

Upload these files to `/Volumes/workspace/bronze/tseries/`:

- `PdM_telemetry.csv`
- `PdM_errors.csv`
- `PdM_failures.csv`
- `PdM_machines.csv`
- `PdM_maint.csv`

In [0]:
paths = {
    "telemetry": f"{VOLUME}/PdM_telemetry.csv",
    "errors": f"{VOLUME}/PdM_errors.csv",
    "failures": f"{VOLUME}/PdM_failures.csv",
    "machines": f"{VOLUME}/PdM_machines.csv",
    "maint": f"{VOLUME}/PdM_maint.csv",
}

for name, path in paths.items():
    print(name, path)

In [0]:
# Read raw CSVs. Infer schema first so the notebook works with the supplied files.
raw = {}
for name, path in paths.items():
    raw[name] = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(path)
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_file", F.lit(path))
    )
    print(name, raw[name].count(), raw[name].columns)

## 1. Bronze MERGE helper
The key idea is **not** to blindly overwrite a Delta table. We merge the incoming batch into the existing Delta table. If the business key already exists, it is updated; otherwise it is inserted.

In [0]:
def merge_to_bronze(df, table_name, keys):
    full = f"{BRONZE}.{table_name}"
    if not spark.catalog.tableExists(full):
        # First load creates the Delta target.
        df.write.format("delta").mode("overwrite").saveAsTable(full)
        print("Created:", full)
        return

    target = DeltaTable.forName(spark, full)
    condition = " AND ".join([f"t.`{k}` <=> s.`{k}`" for k in keys])

    (
        target.alias("t")
        .merge(df.alias("s"), condition)
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print("MERGED:", full)

In [0]:
# Business keys for the supplied predictive-maintenance data.
# Telemetry: machineID + datetime
# Errors: machineID + datetime + errorID
# Failures: machineID + datetime + failure type
# Maintenance: machineID + datetime + component
# Machines: machineID
merge_to_bronze(raw["telemetry"], "telemetry", ["machineID", "datetime"])
merge_to_bronze(raw["errors"], "errors", ["machineID", "datetime", "errorID"])
merge_to_bronze(raw["failures"], "failures", ["machineID", "datetime", "failure"])
merge_to_bronze(raw["maint"], "maint", ["machineID", "datetime", "comp"])
merge_to_bronze(raw["machines"], "machines", ["machineID"])

## 2. Validate Bronze

These checks demonstrate that the Bronze layer contains Delta tables and that repeated execution is intended to be idempotent rather than producing duplicate business records.

In [0]:
for t in ["telemetry","errors","failures","maint","machines"]:
    full = f"{BRONZE}.{t}"
    print(full)
    display(spark.sql(f"DESCRIBE DETAIL {full}"))
    display(spark.table(full).limit(5))

In [0]:
# Duplicate-key checks
checks = {
    "telemetry": ["machineID","datetime"],
    "errors": ["machineID","datetime","errorID"],
    "failures": ["machineID","datetime","failure"],
    "maint": ["machineID","datetime","comp"],
    "machines": ["machineID"],
}
for t, keys in checks.items():
    cols = ", ".join([f"`{k}`" for k in keys])
    dup = spark.table(f"{BRONZE}.{t}").groupBy(*keys).count().filter("count > 1").count()
    print(t, "duplicate business keys =", dup)

### Bronze layer complete

Downstream notebooks should read **only the Bronze Delta tables**, not the CSVs directly.